# Cert Prep — 07: Pandas API on Spark

**Exam weight: ~5%**

Topics covered:
- pyspark.pandas vs pandas
- Creating Pandas-on-Spark DataFrames
- Operations: same as pandas syntax
- toPandas() — driver memory warning
- to_spark() — convert back to Spark DataFrame
- Index handling


In [1]:
import os, time, pathlib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER = 'spark://spark-master:7077'

spark = (
    SparkSession.builder
    .appName('cert-prep')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.sql.warehouse.dir', '/workspace/data/training_warehouse')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
# Required for pyspark.pandas with pyarrow >= 2.0
import os; os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print(f"Spark {spark.version}")

# Sample dataset used throughout
data = [
    (1, "Alice",  "Engineering", 95000.0, "2021-03-15", 4),
    (2, "Bob",    "Marketing",   72000.0, "2020-07-01", 3),
    (3, "Carol",  "Engineering", 105000.0,"2019-11-20", 5),
    (4, "Dave",   "Marketing",   68000.0, "2022-01-10", 2),
    (5, "Eve",    "Engineering", 88000.0, "2021-09-05", 4),
    (6, "Frank",  "HR",          61000.0, "2020-04-18", 3),
    (7, "Grace",  "HR",          None,    "2023-02-28", 1),
    (8, "Hank",   "Engineering", 112000.0,"2018-06-12", 5),
    (9, "Iris",   "Marketing",   None,    "2022-11-03", 2),
    (10,"Jack",   "HR",          59000.0, "2023-08-01", 1),
]
schema = StructType([
    StructField("id",         IntegerType(),  False),
    StructField("name",       StringType(),   True),
    StructField("dept",       StringType(),   True),
    StructField("salary",     DoubleType(),   True),
    StructField("hire_date",  StringType(),   True),
    StructField("perf_score", IntegerType(),  True),
])
df = spark.createDataFrame(data, schema)
df.cache().count()
print("Dataset ready")
df.show()

Spark 4.0.2


Dataset ready
+---+-----+-----------+--------+----------+----------+
| id| name|       dept|  salary| hire_date|perf_score|
+---+-----+-----------+--------+----------+----------+
|  1|Alice|Engineering| 95000.0|2021-03-15|         4|
|  2|  Bob|  Marketing| 72000.0|2020-07-01|         3|
|  3|Carol|Engineering|105000.0|2019-11-20|         5|
|  4| Dave|  Marketing| 68000.0|2022-01-10|         2|
|  5|  Eve|Engineering| 88000.0|2021-09-05|         4|
|  6|Frank|         HR| 61000.0|2020-04-18|         3|
|  7|Grace|         HR|    NULL|2023-02-28|         1|
|  8| Hank|Engineering|112000.0|2018-06-12|         5|
|  9| Iris|  Marketing|    NULL|2022-11-03|         2|
| 10| Jack|         HR| 59000.0|2023-08-01|         1|
+---+-----+-----------+--------+----------+----------+



In [2]:
# ── Pandas API on Spark ───────────────────────────────────────────────────────
print("=== Pandas API on Spark ===")
import pyspark.pandas as ps

# Convert Spark DF to Pandas-on-Spark DF
psdf = df.pandas_api()  # Spark 3.2+ — ps.from_dataframe() does not exist

print(type(psdf))
print(psdf.dtypes)
print()

# Pandas-style operations
print("Head:")
print(psdf.head(3))
print()
print("Describe:")
print(psdf[["salary","perf_score"]].describe())

=== Pandas API on Spark ===
<class 'pyspark.pandas.frame.DataFrame'>
id              int32
name           object
dept           object
salary        float64
hire_date      object
perf_score      int32
dtype: object

Head:
   id   name         dept    salary   hire_date  perf_score
2   3  Carol  Engineering  105000.0  2019-11-20           5
3   4   Dave    Marketing   68000.0  2022-01-10           2
6   7  Grace           HR       NaN  2023-02-28           1

Describe:


[Stage 9:>                                                          (0 + 4) / 4]

              salary  perf_score
count       8.000000   10.000000
mean    82500.000000    3.000000
std     20346.989949    1.490712
min     59000.000000    1.000000
25%     61000.000000    2.000000
50%     72000.000000    3.000000
75%     95000.000000    4.000000
max    112000.000000    5.000000


In [3]:
# ── Operations ───────────────────────────────────────────────────────────────
print("=== Pandas-on-Spark Operations ===")
import pyspark.pandas as ps

psdf = df.pandas_api()

# groupby — same as pandas
print("GroupBy:")
print(psdf.groupby("dept")["salary"].mean())

# fillna
print("\nfillna:")
print(psdf["salary"].fillna(0).sum())

# value_counts
print("\nvalue_counts:")
print(psdf["dept"].value_counts())

# apply — runs as Pandas UDF internally
print("\napply (row-level):")
psdf["salary_k"] = psdf["salary"].apply(lambda x: round(x/1000, 1) if x else 0)
print(psdf[["name","salary","salary_k"]].head())

=== Pandas-on-Spark Operations ===
GroupBy:
dept
Marketing       70000.0
Engineering    100000.0
HR              60000.0
Name: salary, dtype: float64

fillna:
660000.0

value_counts:
dept
Engineering    4
Marketing      3
HR             3
Name: count, dtype: int64

apply (row-level):


[Stage 25:>                                                         (0 + 4) / 4]

    name    salary  salary_k
0  Alice   95000.0      95.0
1    Bob   72000.0      72.0
4    Eve   88000.0      88.0
5  Frank   61000.0      61.0
2  Carol  105000.0     105.0


In [4]:
# ── toPandas() WARNING ───────────────────────────────────────────────────────
print("=== toPandas() Memory Warning ===")
print()
print("df.toPandas() — converts Spark DataFrame to pandas DataFrame")
print("  - Collects ALL data to the DRIVER memory")
print("  - If data is larger than driver memory → OutOfMemoryError")
print("  - Fine for small result sets, DANGEROUS for large DataFrames")
print()
print("Same applies to:")
print("  psdf.to_pandas()   — Pandas-on-Spark → pandas")
print("  psdf.toPandas()    — same")
print()
print("Safe pattern: filter/aggregate FIRST, then toPandas():")
print("  df.groupBy('dept').agg(avg('salary')).toPandas()  # small result ✅")
print("  df.toPandas()                                      # 100M rows ❌")
print()

# Safe usage
small_result = df.groupBy("dept").agg(F.avg("salary").alias("avg")).toPandas()
print("Small result toPandas:")
print(small_result)

=== toPandas() Memory Warning ===

df.toPandas() — converts Spark DataFrame to pandas DataFrame
  - Collects ALL data to the DRIVER memory
  - If data is larger than driver memory → OutOfMemoryError
  - Fine for small result sets, DANGEROUS for large DataFrames

Same applies to:
  psdf.to_pandas()   — Pandas-on-Spark → pandas
  psdf.toPandas()    — same

Safe pattern: filter/aggregate FIRST, then toPandas():
  df.groupBy('dept').agg(avg('salary')).toPandas()  # small result ✅
  df.toPandas()                                      # 100M rows ❌

Small result toPandas:
          dept       avg
0    Marketing   70000.0
1  Engineering  100000.0
2           HR   60000.0


In [5]:
# ── to_spark() — convert back ─────────────────────────────────────────────────
print("=== to_spark() ===")
import pyspark.pandas as ps

psdf = df.pandas_api()

# Apply pandas operations
psdf_filtered = psdf[psdf["salary"] > 80000]

# Convert back to Spark DataFrame for further Spark operations
# index_col must be a NEW name — cannot overlap with existing columns
spark_df = psdf_filtered.to_spark(index_col="row_index")
print(type(spark_df))
spark_df.show()
print()
print("Index column warning:")
print("  Pandas-on-Spark adds a default index column")
print("  When converting to Spark, index becomes a column unless dropped:")
print("  psdf.to_spark(index_col='my_index')")
print("  psdf.to_spark().drop('__index_level_0__')  # drop default index")

=== to_spark() ===
<class 'pyspark.sql.classic.dataframe.DataFrame'>
+---------+---+-----+-----------+--------+----------+----------+
|row_index| id| name|       dept|  salary| hire_date|perf_score|
+---------+---+-----+-----------+--------+----------+----------+
|        0|  1|Alice|Engineering| 95000.0|2021-03-15|         4|
|        2|  3|Carol|Engineering|105000.0|2019-11-20|         5|
|        4|  5|  Eve|Engineering| 88000.0|2021-09-05|         4|
|        7|  8| Hank|Engineering|112000.0|2018-06-12|         5|
+---------+---+-----+-----------+--------+----------+----------+


Index column warning:
  Pandas-on-Spark adds a default index column
  When converting to Spark, index becomes a column unless dropped:
  psdf.to_spark(index_col='my_index')
  psdf.to_spark().drop('__index_level_0__')  # drop default index
